In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import tensorflow as tf
from tensorflow import keras
from keras.models import Model, Sequential
from keras.layers import Input, Conv1D, MaxPooling1D, Dense, Dropout, Flatten, Concatenate, LSTM, BatchNormalization, SpatialDropout1D, GlobalAveragePooling1D
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.losses import Huber
import os
import random
from ProfitStrategy import riskless_profit, profits_summary
from sklearn.utils.class_weight import compute_class_weight
#import tensorflow_addons as tfa

In [2]:
folder = "exported_csvs"

data = {
    os.path.splitext(f)[0]: pd.read_csv(os.path.join(folder, f))
    for f in os.listdir(folder) if f.endswith(".csv")
}

In [3]:
data['back_VWAP_log_return_sign'] = (data['back_VWAP_log_return']>=0).astype(int)
data['lay_VWAP_log_return_sign'] = (data['lay_VWAP_log_return']>=0).astype(int)

In [4]:
train_idx, val_idx, test_idx = np.array(data['train_indices']).ravel(), np.array(data['val_indices']).ravel() ,np.array(data['test_indices']).ravel()

back_features = [
    'back_prices_0', 
    #'back_prices_1',
    #'back_volumes_0', 'back_volumes_1',
    'back_VWAP', 
    #'others_back_PVT', 
    #'back_PVT',
    #'WOM', 
    #'back_VWAP_log_return',
    'implied_win_prob'
]
lay_features = [
    'lay_prices_0', 
    #'lay_prices_1', 
    #'lay_volumes_0','lay_volumes_1',
    #'lay_VWAP', 
    #'others_lay_PVT', 
    #'lay_PVT',
    #'WOM', 
    #'lay_VWAP_log_return',
    #'implied_lose_prob'
]

back_target_feature = 'back_VWAP_log_return_sign'
lay_target_feature = 'lay_VWAP_log_return_sign'

win_target_feature = 'win'
lose_target_feature = 'lose'

commisions = 0.05

cur_back, cur_lay = np.array( data['back_VWAP'].iloc[test_idx, -2]), np.array( data['lay_VWAP'].iloc[test_idx, -2] )
future_back, future_lay = np.array( data['back_VWAP'].iloc[test_idx, -1] ), np.array( data['lay_VWAP'].iloc[test_idx, -1] )

perfect_riskless_profit = np.array([
    riskless_profit(cb, cl, fb, fl, fob, fol)
    for cb, cl, fb, fl, fob, fol in zip(
        cur_back, cur_lay,
        future_back, future_lay,
        future_back, future_lay
    )
])

def profit_summary(x):
    return profits_summary(x, perfect_riskless_profit)

In [5]:
SEED = 5

def set_seeds(seed=SEED):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    tf.random.set_seed(seed)
    np.random.seed(seed)
    
def set_global_determinism(seed=SEED):
    set_seeds(seed=seed)

    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
    
    tf.config.threading.set_inter_op_parallelism_threads(1)
    tf.config.threading.set_intra_op_parallelism_threads(1)

In [6]:
def Prepare_Data(selected_features, target_feature, dict_data = data, 
                 train_idx = train_idx, val_idx = val_idx, test_idx = test_idx):
    
    X = np.stack([dict_data[feat].iloc[:, :-1] for feat in selected_features], axis=-1)
    y = np.array(dict_data[target_feature].iloc[:, -1]).reshape(-1, 1)

    X_train, X_val, X_test = X[train_idx], X[val_idx], X[test_idx]
    y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]
    
    scaler_X = StandardScaler()
    
    for i in range(X_train.shape[-1]):
        X_train[:,:,i] = scaler_X.fit_transform(X_train[:,:,i])
        X_val[:,:,i] = scaler_X.transform(X_val[:,:,i])
        X_test[:,:,i] = scaler_X.transform(X_test[:,:,i])

    return X_train, X_val, X_test, y_train, y_val, y_test.ravel()

In [7]:
def CNN_LSTM_Model(input_shape=(9,len(back_features))):
    model = Sequential([
        Input(shape=input_shape),
        
        Conv1D(filters=64, kernel_size=3, 
               #padding='same', 
               activation='relu'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        SpatialDropout1D(0.2),
        
        Conv1D(filters=32, kernel_size=3, 
               #padding='same', 
               activation='relu'),
        BatchNormalization(),
        SpatialDropout1D(0.2),
        
        LSTM(64, return_sequences=True),
        LSTM(32, return_sequences=False),
    
        #GlobalAveragePooling1D(),
        Dense(50, activation='relu'),
        Dropout(0.3),
        Dense(25, activation='relu'),
        Dense(1, activation='sigmoid')
        #Dense(1)
    ])

    def soft_sign_loss(y_true, y_pred):
    # Loss = large when y_true and y_pred have opposite signs
    # Use: - y_true * y_pred — positive when signs match, negative when they don't
        return tf.reduce_mean(tf.nn.relu(-y_true * y_pred))


    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss= 'binary_crossentropy'
        #loss = Huber(delta=1.0)
        #loss=tfa.losses.SigmoidFocalCrossEntropy()
    )

    #model.summary()
    
    return model

def evaluate(y_test, y_pred):
    #mse = mean_squared_error(y_test, y_pred)
    #mae = mean_absolute_error(y_test, y_pred)
    #r2 = r2_score(y_test, y_pred)
    #mape = mean_absolute_percentage_error(y_test, y_pred)*100
    #rmse = np.sqrt(mse)
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    accuracy = accuracy_score(y_test, y_pred)
    
    metrics = {
        #'MSE': mse,
        #'RMSE': rmse,
        #'MAE': mae,
        #'MAPE': mape,
        #'R2': r2,
        'f1_score': f1,
        'precision_score' : precision,
        'recall_score': recall,
        'accuracy_score': accuracy
    }
    return metrics

In [8]:
def best_threshold(y_val_true, y_val_probs, r = 0.01):
    thresholds = np.arange(0, 1.01, 0.01)
    scores = []

    for t in thresholds:
        y_val_pred = (y_val_probs >= t).astype(int)
        recall = recall_score(y_val_true, y_val_pred)
        precision = precision_score(y_val_true, y_val_pred) if (y_val_pred.sum() > 0) else 0

        if recall >= r:
            score = precision
        else:
            score = -np.inf

        scores.append(score)

    best_t = thresholds[np.argmax(scores)]
    print('val_set best precision:', max(scores))
    
    return best_t

def CNN_LSTM_Results(selected_features, target_feature, dict_data = data, 
              train_idx = train_idx, test_idx = test_idx,
              epochs=100, batch_size=32, verbose=0):
    
    set_global_determinism(seed=SEED)
    
    model = CNN_LSTM_Model( input_shape = (9, len(selected_features) ) )

    X_train_scaled, X_val_scaled, X_test_scaled, y_train, y_val, y_test = \
    Prepare_Data(selected_features, target_feature)

    callbacks = [
            EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-7)
        ]
    
    class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train.ravel())

    weight_dict = {0: class_weights[0], 1: class_weights[1]}

    model.fit(X_train_scaled, y_train, epochs=epochs, batch_size=batch_size,
              class_weight=weight_dict,
              #validation_split=0.2,
              validation_data=(X_val_scaled, y_val),
              callbacks=callbacks,verbose=verbose)

    y_val_probs = model.predict(X_val_scaled).ravel()

    t = best_threshold( y_val.ravel() , y_val_probs)

    y_test_probs = model.predict(X_test_scaled).ravel()

    y_pred = (y_test_probs >= t).astype(int)
    
    return {'forecast':y_pred, 
            'model':model, 
            'test_true':y_test,
            'metrics': evaluate(y_test, y_pred),
            'val_probs':y_val_probs,
            'test_probs':y_test_probs}

In [9]:
#back_results = CNN_LSTM_Results(back_features, back_target_feature)
#lay_results = CNN_LSTM_Results(lay_features, lay_target_feature)
#print( back_results['metrics'] )
#print( lay_results['metrics'] )

In [10]:
win_results = CNN_LSTM_Results(back_features, win_target_feature, cur_back)
lose_results = CNN_LSTM_Results(lay_features, lose_target_feature, cur_lay)

270/270 ━━━━━━━━━━━━━━━━━━━━ 0s 760us/step
270/270 ━━━━━━━━━━━━━━━━━━━━ 0s 758us/step


In [11]:
test_race_ids = data['race_id'].iloc[:,-1][test_idx]

win = pd.DataFrame({
    'race_id': test_race_ids,
    'y_true': win_results['test_true'],
    'y_prob': win_results['test_probs']
})

lose = pd.DataFrame({
    'race_id': test_race_ids,
    'y_true': lose_results['test_true'],
    'y_prob': lose_results['test_probs']
})

def topwin_k_accuracy(df, k=3):
    return df.groupby('race_id').apply(
                lambda g: g.nlargest(k, 'y_prob')['y_true'].max(), include_groups=False).mean()
def toplose_k_accuracy(df, k=3):
    return df.groupby('race_id').apply(
                lambda g: g.nlargest(k, 'y_prob')['y_true'].min(), include_groups=False).mean()

In [12]:
print('win:')

top1 = topwin_k_accuracy(win, k=1)
top3 = topwin_k_accuracy(win, k=3)
top5 = topwin_k_accuracy(win, k=5)

print(f"Top-1 Accuracy: {top1:.4f}")
print(f"Top-3 Accuracy: {top3:.4f}")
print(f"Top-5 Accuracy: {top5:.4f}")
    
print('lose:')

top1 = toplose_k_accuracy(lose, k=1)
top3 = toplose_k_accuracy(lose, k=3)
top5 = toplose_k_accuracy(lose, k=5)

print(f"Top-1 Accuracy: {top1:.4f}")
print(f"Top-3 Accuracy: {top3:.4f}")
print(f"Top-5 Accuracy: {top5:.4f}")

win:
Top-1 Accuracy: 0.3546
Top-3 Accuracy: 0.6866
Top-5 Accuracy: 0.8619
lose:
Top-1 Accuracy: 0.9638
Top-3 Accuracy: 0.8315
Top-5 Accuracy: 0.5975


In [13]:
win_odds = pd.DataFrame({
    'race_id': test_race_ids,
    'y_true': win_results['test_true'],
    'y_prob': data['implied_win_prob'].iloc[:,-1][test_idx]
})

print('win_odds:')

top1 = topwin_k_accuracy(win_odds, k=1)
top3 = topwin_k_accuracy(win_odds, k=3)
top5 = topwin_k_accuracy(win_odds, k=5)

print(f"Top-1 Accuracy: {top1:.4f}")
print(f"Top-3 Accuracy: {top3:.4f}")
print(f"Top-5 Accuracy: {top5:.4f}")

win_odds:
Top-1 Accuracy: 0.3702
Top-3 Accuracy: 0.7013
Top-5 Accuracy: 0.8717


In [14]:
lose_odds = pd.DataFrame({
    'race_id': test_race_ids,
    'y_true': lose_results['test_true'],
    'y_prob': data['implied_lose_prob'].iloc[:,-2][test_idx]
})

print('lose_odds:')

top1 = toplose_k_accuracy(lose_odds, k=1)
top3 = toplose_k_accuracy(lose_odds, k=3)
top5 = toplose_k_accuracy(lose_odds, k=5)

print(f"Top-1 Accuracy: {top1:.4f}")
print(f"Top-3 Accuracy: {top3:.4f}")
print(f"Top-5 Accuracy: {top5:.4f}")

lose_odds:
Top-1 Accuracy: 0.9667
Top-3 Accuracy: 0.8315
Top-5 Accuracy: 0.6014
